In [ ]:
import numpy as np
import re
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import meme

from tqdm import tqdm

In [ ]:
start=datetime(2021,1,1,0)
end=datetime(2022,10,13,0)
# end=datetime.now()
times=[start];

while times[-1]<end:
    times.append(times[-1]+timedelta(hours=4)); 

In [ ]:
KAct=meme.names.list_pvs('USEG:UNDS:%:KAct')
PVs=KAct
# PVs

2026-05-07T10:33:11.696722000 WARN pvxs.client.io Server 134.79.151.36:42467 no supported auth.  try to force 'anonymous'


In [ ]:
import ipykernel
ipykernel.__version__

'6.29.5'

In [ ]:
import sys
sys.version

'3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]'

In [ ]:
# pull undulator K values from the archive for each time in times
# first restore any cached rows from kvals.csv so we only fetch missing times
# use one threadpool whose workers draw from missing times; advance the progress bar as jobs finish
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from tqdm.auto import tqdm

def fetch_kvals(ii, ft):
    archive_data = meme.archive.get(PVs, from_time=ft, to_time=ft+timedelta(seconds=3), timeout=100)
    Ks = np.array([x['value']['value']['values'][-1] for x in archive_data if x is not None])
    return ii, ft, archive_data, Ks

cache_path = Path('kvals.csv')
cached_kvals_by_time = {}
cached_und_cells = None

if cache_path.exists():
    lines = [line.strip() for line in cache_path.read_text().splitlines() if line.strip()]
    if lines:
        header = lines[0].split(',')
        if len(header) > 1:
            cached_und_cells = np.array([float(value) for value in header[1:]])
        for line in lines[1:]:
            parts = line.split(',')
            if len(parts) < 2:
                continue
            try:
                dt = datetime.strptime(parts[0], '%Y-%m-%d %H:%M:%S')
                kval = np.array([float(value) for value in parts[1:]])
            except ValueError:
                continue
            if cached_und_cells is not None and len(kval) != len(cached_und_cells):
                continue
            cached_kvals_by_time[dt] = kval

requested_times = list(times)
results = [None] * len(requested_times)
archive_data = None
missing = []

for ii, ft in enumerate(requested_times):
    cached_kvals = cached_kvals_by_time.get(ft)
    if cached_kvals is not None:
        results[ii] = (ft, cached_kvals)
    else:
        missing.append((ii, ft))

if missing:
    with ThreadPoolExecutor(max_workers=min(8, len(missing) or 1)) as executor:
        futures = [executor.submit(fetch_kvals, ii, ft) for ii, ft in missing]
        for future in tqdm(as_completed(futures), total=len(futures), desc='Fetching K values'):
            try:
                ii, ft, fetched_archive_data, Ks = future.result()
                results[ii] = (ft, Ks)
                cached_kvals_by_time[ft] = Ks
                if archive_data is None:
                    archive_data = fetched_archive_data
            except Exception:
                pass
                # print(f'Error fetching data for time {ft}: {exc}')

pairs = [result for result in results if result is not None]
times = [ft for ft, _ in pairs]
kvals = [kv for _, kv in pairs]
all_kvals_by_time = dict(sorted(cached_kvals_by_time.items()))

Fetching K values:   0%|          | 0/2197 [00:00<?, ?it/s]

Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct faile

124.00s - thread._ident is None in _get_related_thread!


Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range


1.92s - thread._ident is None in _get_related_thread!


Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range


0.93s - thread._ident is None in _get_related_thread!


Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct faile

3.74s - thread._ident is None in _get_related_thread!


Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2350:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2250:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2450:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2150:KAct failed due to error: list index out of range
Request for PV USEG:UNDS:2550:KAct faile

In [ ]:
if archive_data is not None:
    und_cells=np.array([float(re.search(':([0-9][0-9])[0-9][0-9]:',x['pvName'])[1]) for x in archive_data if x is not None])
elif cached_und_cells is not None:
    und_cells=cached_und_cells
else:
    raise RuntimeError('Could not determine undulator cells from archive data or kvals.csv cache')

In [ ]:
# rewrite kvals.csv with the union of cached and newly fetched rows
# keeping rows in chronological order means new data is appended or prepended naturally

all_kvals_by_time.update({dt: k for dt, k in zip(times, kvals)})

with open('kvals.csv', 'w') as f:
    f.write('Datetime,' + ','.join(map(str, und_cells)) + '\n')
    for dt in sorted(all_kvals_by_time):
        kval = all_kvals_by_time[dt]
        f.write(dt.strftime('%Y-%m-%d %H:%M:%S') + ',' + ','.join(map(str, kval)) + '\n')

Writing kvals: 100%|██████████| 3901/3901 [00:00<00:00, 35205.15it/s]


In [ ]:
from scipy.ndimage import binary_erosion,binary_dilation
def lasing_und(Ks,und_cells,rho=3e-3,rhoscale=4,verbose=False):
    kdist=rho*rhoscale
    kidxs=np.argsort(Ks)
    ksort=Ks[kidxs]
    mask=(np.abs(np.diff(ksort,prepend=-1))/ksort<kdist)|(np.abs(np.diff(ksort,append=-1))/ksort<kdist) #alls undulators who have another undulator near them in K
    mask=mask&(ksort>0.75) # Get rid of extracted undulators
    fund_idxs=kidxs[mask] #fund is for fundamental, to be distinguished from harmonics (considered later)
    
    # We should throw away any "group" of Ks with less than 5 undulators in it.
    kmin_idx=np.where(~(np.abs(np.diff(ksort,prepend=-1))/ksort<kdist)&(np.abs(np.diff(ksort,append=-1))/ksort<kdist) )[0] #idx of the lowest K in "group" of close Ks.
    #print(ksort[kmin_idx])
    for  kmin,kmax in zip(kmin_idx, list(kmin_idx[1:])+[len(kidxs)]):
        if np.sum(mask[kmin:kmax])<6:
            mask[kmin:kmax]=0
    #print(ksort[mask])
    #print(und_cells[kidxs][mask])
    fund_idxs=kidxs[mask] #fund is for fundamental, to be distinguished from harmonics (considered later)
    
    lasing_cells=und_cells[fund_idxs]
    lasing_ks=Ks[fund_idxs]
    idxs=np.argsort(lasing_cells)
    lasing_cells=lasing_cells[idxs]
    lasing_ks=lasing_ks[idxs]
    
    
    ## Check for xleap
    #Look for 4 consecutive reverse tapered undulators, with reverse taper rate larger than rho. Keep everything within that band.
    if len(lasing_cells)>4:
        if verbose: print(lasing_cells)
        mask= (np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks>(rho/2)) & (np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks<(1.5*kdist))
        if verbose: print(np.diff(lasing_ks,append=lasing_ks[-1])/lasing_ks)
        if verbose: print(mask)
        mask=binary_dilation(binary_erosion(mask,structure=[1, 1, 1, 1]),structure=[1, 1, 1, 1])
        if verbose: print(mask)
    else:
        mask=[0]
        lasing_cells=und_cells[0]
        lasing_ks=Ks[0]
    if sum(mask)>0:
        if (np.max(lasing_cells[mask])-np.min(lasing_cells[mask]))>6:
            mask=mask*0
    if sum(mask)>0:
        xleap=True
        taper_rate,taper_rate_std=(np.median(np.diff(lasing_ks[mask]))/np.mean(lasing_ks[mask]),np.std(np.diff(lasing_ks[mask]))/np.mean(lasing_ks[mask]))
        taper_ks=lasing_ks[mask]
        taper_cells=lasing_cells[mask]
        kmin=lasing_ks[mask][0]-kdist*lasing_ks[mask][0]
        kmax=lasing_ks[mask][-1]+kdist*lasing_ks[mask][-1]
        mask=(lasing_ks>kmin)&(lasing_ks<kmax)
        #diffmask=(np.abs(np.diff(lasing_ks,prepend=-1))/lasing_ks<kdist)|(np.abs(np.diff(lasing_ks,append=-1))/lasing_ks<kdist)
        #mask=mask*diffmask
    else:
        xleap=False
        taper_rate,taper_rate_std = 0,0
        mask=np.ones(np.shape(lasing_ks))==1
        taper_ks=[]
        taper_cells=[]

    data={'xleap':xleap,'taper_rate':taper_rate,'taper_rate_std':taper_rate_std,'lasing_cells':lasing_cells[mask],'lasing_ks':lasing_ks[mask],
          'taper_cells':taper_cells,'taper_ks':taper_ks}
    return data

lasing_und(Ks,und_cells,kdist=1e-2)
unds=np.array([float(re.search(':([0-9][0-9])[0-9][0-9]:',x['pvName'])[1]) for x in archive_data])
lasing_und(Ks,unds)

fig = plt.figure(num=1,figsize=[3.375*1.61*2,3.375])
ax1=plt.subplot(1,2,1)
ax2=plt.subplot(1,2,2)
datas=[];
for ii,(time,kv) in enumerate(zip(times,kvals)):
    data=lasing_und(kv,unds)
    lasing_cells=data['lasing_cells']
    lasing_ks=data['lasing_ks']
    if data['xleap']:
        datas.append(data)
        datas[-1]['time']=time
        datas[-1]['idx']=ii
        #ax1.plot(lasing_cells-lasing_cells[0],(lasing_ks-lasing_ks[0])/np.median(lasing_ks))
        ax1.plot(data['taper_cells'],(data['taper_ks']-data['taper_ks'][0])/np.median(data['taper_ks']))
    else:
        ax2.plot(lasing_cells-lasing_cells[0],(lasing_ks-lasing_ks[0])/np.median(lasing_ks))

NameError: name 'Ks' is not defined